In [1]:
import pandas as pd
import re
original_df = pd.read_json("../02-raw/k8s_incidents_transformed.jsonl", lines=True)

original_df.head()

,scenario_id,pod_describe,pod_logs,pod_logs_previous
0,createcontainerconfigerror_missing_secret,Name: service-w-bm4b-0\nNamespace:...,"Error from server (BadRequest): container ""mai...",
1,createcontainerconfigerror_missing_secret,Name: service-w-cmbj-69b87b95bd-4v...,"Error from server (BadRequest): container ""ser...",
2,createcontainerconfigerror_missing_secret,Name: backend-w-ks1x-0\nNamespace:...,"Error from server (BadRequest): container ""app...",
3,createcontainerconfigerror_missing_secret,Name: api-w-s3zf-7b6946fdc-vw8gw\n...,"Error from server (BadRequest): container ""app...",
4,createcontainerconfigerror_missing_secret,Name: service-w-01zx-0\nNamespace:...,"Error from server (BadRequest): container ""mai...",


In [2]:
df = original_df[["scenario_id", "pod_describe", "pod_logs", "pod_logs_previous"]].copy()

In [3]:
df.head()

,scenario_id,pod_describe,pod_logs,pod_logs_previous
0,createcontainerconfigerror_missing_secret,Name: service-w-bm4b-0\nNamespace:...,"Error from server (BadRequest): container ""mai...",
1,createcontainerconfigerror_missing_secret,Name: service-w-cmbj-69b87b95bd-4v...,"Error from server (BadRequest): container ""ser...",
2,createcontainerconfigerror_missing_secret,Name: backend-w-ks1x-0\nNamespace:...,"Error from server (BadRequest): container ""app...",
3,createcontainerconfigerror_missing_secret,Name: api-w-s3zf-7b6946fdc-vw8gw\n...,"Error from server (BadRequest): container ""app...",
4,createcontainerconfigerror_missing_secret,Name: service-w-01zx-0\nNamespace:...,"Error from server (BadRequest): container ""mai...",


In [4]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none"}:
        return ""
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    lines = [line.rstrip() for line in x.split("\n")]
    return "\n".join(lines).strip()

In [5]:
for col in ["pod_describe", "pod_logs", "pod_logs_previous"]:
    df[col] = df[col].apply(normalize_text)

In [6]:
def extract_first(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
    if not text:
        return None
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

In [7]:
def extract_block(text, block_name):
    if not text:
        return None

    lines = text.split("\n")
    block = []
    in_block = False

    for line in lines:
        if not in_block:
            if re.match(rf"^{re.escape(block_name)}:\s*$", line.strip()):
                in_block = True
            continue

        if line and not line.startswith(" ") and re.match(r"^[A-Za-z0-9_.\-/ ]+:\s*", line):
            break

        block.append(line)

    out = "\n".join(block).strip()
    return out if out else None

In [8]:
def parse_events_block(events_text):
    if not events_text:
        return {
            "event_reason": None,
            "event_message": None,
            "all_event_reasons": [],
            "all_event_messages": []
        }

    reasons = []
    messages = []

    lines = [line for line in events_text.split("\n") if line.strip()]

    for line in lines:
        if "Reason" in line and "Message" in line:
            continue
        if line.strip().startswith("----"):
            continue

        parts = re.split(r"\s{2,}", line.strip())
        if len(parts) >= 5:
            reasons.append(parts[1].strip())
            messages.append(parts[-1].strip())

    return {
        "event_reason": reasons[0] if reasons else None,
        "event_message": messages[0] if messages else None,
        "all_event_reasons": reasons,
        "all_event_messages": messages
    }

In [9]:
def parse_describe(text):
    containers_block = extract_block(text, "Containers")
    volumes_block = extract_block(text, "Volumes")
    tolerations_block = extract_block(text, "Tolerations")
    events_block = extract_block(text, "Events")

    events = parse_events_block(events_block)

    return {
        "pod_name": extract_first(r"^Name:\s*(.+)$", text),
        "namespace": extract_first(r"^Namespace:\s*(.+)$", text),
        "service_account_name": extract_first(r"^Service Account:\s*(.+)$", text),
        "node": extract_first(r"^Node:\s*(.+)$", text),
        "pod_status": extract_first(r"^Status:\s*(.+)$", text),
        "pod_ip": extract_first(r"^IP:\s*(.+)$", text),
        "controlled_by": extract_first(r"^Controlled By:\s*(.+)$", text),
        "image": extract_first(r"^\s*Image:\s*(.+)$", containers_block or text),
        "container_state": extract_first(r"^\s*State:\s*(.+)$", containers_block or text),
        "last_state": extract_first(r"^\s*Last State:\s*(.+)$", containers_block or text),
        "ready": extract_first(r"^\s*Ready:\s*(.+)$", containers_block or text),
        "restart_count": extract_first(r"^\s*Restart Count:\s*(.+)$", containers_block or text),
        "node_selectors": extract_first(r"^Node-Selectors:\s*(.+)$", text),
        "qos_class": extract_first(r"^QoS Class:\s*(.+)$", text),
        "claim_name": extract_first(r"^\s*ClaimName:\s*(.+)$", volumes_block or ""),
        "describe_events_raw": events_block,
        "describe_containers_raw": containers_block,
        "describe_volumes_raw": volumes_block,
        "describe_tolerations_raw": tolerations_block,
        "event_reason": events["event_reason"],
        "event_message": events["event_message"],
        "all_event_reasons": events["all_event_reasons"],
        "all_event_messages": events["all_event_messages"],
    }

In [10]:
describe_parsed = df["pod_describe"].apply(parse_describe).apply(pd.Series)
df = pd.concat([df, describe_parsed], axis=1)

In [11]:
def build_evidence_text(row):
    parts = []

    if row.get("pod_describe"):
        parts.append("POD DESCRIBE:\n" + row["pod_describe"])

    if row.get("pod_logs"):
        parts.append("POD LOGS:\n" + row["pod_logs"])

    if row.get("pod_logs_previous"):
        parts.append("POD LOGS PREVIOUS:\n" + row["pod_logs_previous"])

    return "\n\n".join(parts).strip() if parts else None

In [12]:
df["evidence_text"] = df.apply(build_evidence_text, axis=1)

In [13]:
parsed_df = df[
    [
        "scenario_id",
        "namespace",
        "pod_name",
        "service_account_name",
        "node",
        "pod_status",
        "image",
        "container_state",
        "last_state",
        "ready",
        "restart_count",
        "node_selectors",
        "claim_name",
        "event_reason",
        "event_message",
        "evidence_text",
    ]
].copy()

In [14]:
parsed_df.head(20)

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,evidence_text
0,createcontainerconfigerror_missing_secret,team-b-stg-xqqv2a,service-w-bm4b-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-xqqv2a/servic...,POD DESCRIBE:\nName: service-w-bm4...
1,createcontainerconfigerror_missing_secret,orders-prod-sk53ni,service-w-cmbj-69b87b95bd-4vpb4,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:2.0.1,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-sk53ni/servi...,POD DESCRIBE:\nName: service-w-cmb...
2,createcontainerconfigerror_missing_secret,team-b-stg-qkb6mw,backend-w-ks1x-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/backend:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-qkb6mw/backen...,POD DESCRIBE:\nName: backend-w-ks1...
3,createcontainerconfigerror_missing_secret,team-a-dev-0qpcfa,api-w-s3zf-7b6946fdc-vw8gw,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/api:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-a-dev-0qpcfa/api-w-...,POD DESCRIBE:\nName: api-w-s3zf-7b...
4,createcontainerconfigerror_missing_secret,orders-prod-gyn4fj,service-w-01zx-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-gyn4fj/servi...,POD DESCRIBE:\nName: service-w-01z...
5,createcontainerconfigerror_bad_configmap_key,orders-prod-g1sz1z,web-w-a87n-57867b8c9f-b7mcl,default,k8s-incidents-control-plane/172.22.0.2,Pending,docker.io/library/nginx:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-g1sz1z/web-w...,POD DESCRIBE:\nName: web-w-a87n-57...
6,createcontainerconfigerror_bad_configmap_key,team-a-dev-b18b7u,worker-w-qdtk-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/worker:latest,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-a-dev-b18b7u/worker...,POD DESCRIBE:\nName: worker-w-qdtk...
7,createcontainerconfigerror_bad_configmap_key,infra-lab-bzl2st,job-w-84dl-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,docker.io/library/busybox:1.0.0,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned infra-lab-bzl2st/job-w-8...,POD DESCRIBE:\nName: job-w-84dl-0\...
8,createcontainerconfigerror_bad_configmap_key,team-b-stg-45hckl,web-w-t39d-7794967c46-k8js9,default,k8s-incidents-control-plane/172.22.0.2,Pending,docker.io/library/nginx:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-45hckl/web-w-...,POD DESCRIBE:\nName: web-w-t39d-77...
9,createcontainerconfigerror_bad_configmap_key,team-b-stg-o0m6jr,service-w-3bng-6d64d78cf-hxj6b,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:2.0.1,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-o0m6jr/servic...,POD DESCRIBE:\nName: service-w-3bn...


In [15]:
preview_df = parsed_df.copy()

for col in ["event_message", "evidence_text"]:
    if col in preview_df.columns:
        preview_df[col] = preview_df[col].fillna("").str.slice(0, 200)

preview_df

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,evidence_text
0,createcontainerconfigerror_missing_secret,team-b-stg-xqqv2a,service-w-bm4b-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-xqqv2a/servic...,POD DESCRIBE:\nName: service-w-bm4...
1,createcontainerconfigerror_missing_secret,orders-prod-sk53ni,service-w-cmbj-69b87b95bd-4vpb4,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:2.0.1,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-sk53ni/servi...,POD DESCRIBE:\nName: service-w-cmb...
2,createcontainerconfigerror_missing_secret,team-b-stg-qkb6mw,backend-w-ks1x-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/backend:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-qkb6mw/backen...,POD DESCRIBE:\nName: backend-w-ks1...
3,createcontainerconfigerror_missing_secret,team-a-dev-0qpcfa,api-w-s3zf-7b6946fdc-vw8gw,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/api:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-a-dev-0qpcfa/api-w-...,POD DESCRIBE:\nName: api-w-s3zf-7b...
4,createcontainerconfigerror_missing_secret,orders-prod-gyn4fj,service-w-01zx-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-gyn4fj/servi...,POD DESCRIBE:\nName: service-w-01z...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,imagepull_bad_tag,team-b-stg-3twsfm,backend-w-xcuq-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/backend:v99.99.99,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-3twsfm/backen...,POD DESCRIBE:\nName: backend-w-xcu...
1496,imagepull_bad_tag,payments-dev-292goc,worker-w-u6cd-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/worker:0.0.0-nonexistent,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned payments-dev-292goc/work...,POD DESCRIBE:\nName: worker-w-u6cd...
1497,imagepull_bad_tag,infra-lab-i2lqrl,web-w-1yq6-565b469c94-2z7sm,default,k8s-incidents-control-plane/172.22.0.2,Pending,docker.io/library/nginx:0.0.0-nonexistent,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned infra-lab-i2lqrl/web-w-1...,POD DESCRIBE:\nName: web-w-1yq6-56...
1498,imagepull_bad_tag,orders-prod-swjrt0,web-w-7h8q-6496476f7b-p66tp,default,k8s-incidents-control-plane/172.22.0.2,Pending,docker.io/library/nginx:nightly-fake,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-swjrt0/web-w...,POD DESCRIBE:\nName: web-w-7h8q-64...


In [16]:
final_df = df.copy()

In [17]:
cols = [
    "scenario_id",
    "namespace",
    "pod_name",
    "service_account_name",
    "node",
    "pod_status",
    "image",
    "container_state",
    "last_state",
    "ready",
    "restart_count",
    "node_selectors",
    "claim_name",
    "event_reason",
    "event_message",
    "pod_describe",
    "pod_logs",
    "pod_logs_previous",
    "evidence_text",
]

In [18]:
final_df = df[cols].copy()
final_df.head()

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,pod_describe,pod_logs,pod_logs_previous,evidence_text
0,createcontainerconfigerror_missing_secret,team-b-stg-xqqv2a,service-w-bm4b-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-xqqv2a/servic...,Name: service-w-bm4b-0\nNamespace:...,"Error from server (BadRequest): container ""mai...",,POD DESCRIBE:\nName: service-w-bm4...
1,createcontainerconfigerror_missing_secret,orders-prod-sk53ni,service-w-cmbj-69b87b95bd-4vpb4,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:2.0.1,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-sk53ni/servi...,Name: service-w-cmbj-69b87b95bd-4v...,"Error from server (BadRequest): container ""ser...",,POD DESCRIBE:\nName: service-w-cmb...
2,createcontainerconfigerror_missing_secret,team-b-stg-qkb6mw,backend-w-ks1x-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/backend:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-b-stg-qkb6mw/backen...,Name: backend-w-ks1x-0\nNamespace:...,"Error from server (BadRequest): container ""app...",,POD DESCRIBE:\nName: backend-w-ks1...
3,createcontainerconfigerror_missing_secret,team-a-dev-0qpcfa,api-w-s3zf-7b6946fdc-vw8gw,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/api:stable,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned team-a-dev-0qpcfa/api-w-...,Name: api-w-s3zf-7b6946fdc-vw8gw\n...,"Error from server (BadRequest): container ""app...",,POD DESCRIBE:\nName: api-w-s3zf-7b...
4,createcontainerconfigerror_missing_secret,orders-prod-gyn4fj,service-w-01zx-0,default,k8s-incidents-control-plane/172.22.0.2,Pending,ghcr.io/acme/service:1.2.3,Waiting,None,False,0,<none>,None,Scheduled,Successfully assigned orders-prod-gyn4fj/servi...,Name: service-w-01zx-0\nNamespace:...,"Error from server (BadRequest): container ""mai...",,POD DESCRIBE:\nName: service-w-01z...
